In [2]:
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import os, random
import pandas as pd

## Imports for plotting
import matplotlib.pyplot as plt
%matplotlib inline
from matplotlib.colors import to_rgba
import seaborn as sns
sns.set_theme('notebook', style='whitegrid')

## Progress bar
from tqdm.notebook import tqdm

In [3]:
import torch
torch.manual_seed(42) # Setting the seed

print("Using torch", torch.__version__)

Using torch 2.10.0+cpu


In [9]:
df =  pd.read_csv("dataset/toy_dataset_label.csv", sep="\t" ,on_bad_lines="skip")

print(df.head(5))

   ID   FILE            AUTHOR                        BORN-DIED  \
0   1  1.jpg  AACHEN, Hans von  (b. 1552, Köln, d. 1615, Praha)   
1   2  2.jpg  AACHEN, Hans von  (b. 1552, Köln, d. 1615, Praha)   
2   3  3.jpg  AACHEN, Hans von  (b. 1552, Köln, d. 1615, Praha)   
3   4  4.jpg  AACHEN, Hans von  (b. 1552, Köln, d. 1615, Praha)   
4   5  5.jpg  AACHEN, Hans von  (b. 1552, Köln, d. 1615, Praha)   

                                TITLE     DATE                    TECHNIQUE  \
0                            Allegory     1598    Oil on copper, 56 x 47 cm   
1            Bacchus, Ceres and Cupid        -  Oil on canvas, 163 x 113 cm   
2                       Joking Couple        -      Copperplate, 25 x 20 cm   
3       Portrait of Emperor Rudolf II    1590s    Oil on canvas, 60 x 48 cm   
4  Self-Portrait with a Glass of Wine  c. 1596    Oil on canvas, 53 x 44 cm   

                           LOCATION      FORM          TYPE  SCHOOL  \
0           Alte Pinakothek, Munich  painting  myth

In [14]:
techniques = pd.unique(df['TECHNIQUE'])

print(len(list(techniques)))

21437


In [18]:
df['TECHNIQUE_CLEAN'] = df['TECHNIQUE'].str.split(',').str[0]

# 2. Pulizia extra: togliamo spazi bianchi e rendiamo tutto minuscolo
df['TECHNIQUE_CLEAN'] = df['TECHNIQUE_CLEAN'].str.strip().str.lower()

print(len(pd.unique(df['TECHNIQUE_CLEAN'])))

print(f"unique classes: {list(pd.unique(df['TECHNIQUE_CLEAN']))}")

2604
unique classes: ['oil on copper', 'oil on canvas', 'copperplate', 'wood', 'maiolica', 'ceramics', 'ceramic mural composition', '-', 'marble', 'oil on panel', 'oil on cardboard', 'silver', 'engraving', 'wax on obsidian', 'oil and gouache on paper', 'bronze', 'lead', 'drawing', 'terracotta', 'photo', 'white marble', 'fresco', 'oil on oak panel', 'brass', 'cast copper alloy', 'black chalk on vellum', 'stained glass', 'oil on wood', 'oil on oak', 'oil on oak wood', 'oil on paper laid down on board', 'gilt and painted terracotta', 'terracotta with traces of paint', 'glazed terracotta', 'pietra caciolfa', 'pencil and gouache on paper', 'tempera and leaf on panel', 'tempera on panel', 'oil on wood transferred to canvas', 'black crayon with white highlights on blue paper', 'engraving. 320 x 170 cm', 'engraving. 280 x 190 mm', 'panel', 'watercolour', 'woodcut', 'tempera and gold on wood', 'tempera on wood', 'stucco', 'black marble', 'oil on canvas : 118 x 92 cm', 'oil on canvas : 86 x 136 

In [19]:
print(df['TECHNIQUE_CLEAN'].value_counts().head(10)) 

TECHNIQUE_CLEAN
oil on canvas       13657
fresco               4316
oil on panel         3041
photo                2060
oil on wood          1848
marble               1808
tempera on wood      1056
bronze                863
oil on oak panel      740
tempera on panel      699
Name: count, dtype: int64
